# Kaggle Training: LogFilter Tier-2 Transformer

Fine-tunes **cisco-ai/SecureBERT2.0-base** as the tier-2 log-window classifier.

**Before running:**
1. Add the dataset `jacobvalor/hdfs-tracebench-preprocessed-logs` in the right panel
2. Enable GPU T4 x2 in Settings (right panel)
3. Run cells in order — do not skip the sampled verification run

**Expected time:**
- Sampled run (50K normal + 10K failure, 1 epoch): ~15-25 min
- Full run (all data, 2 epochs): ~2-4 hours

## 1. Locate repository

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

# Kaggle compatibility settings must be set before torch/transformers imports.
os.environ.setdefault('TORCHDYNAMO_DISABLE', '1')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

KAGGLE_WORKING = Path('/kaggle/working')
REPO_DIR = Path.cwd()
GIT_URL = 'https://github.com/Jacob-Valor/Modern-AI-Log-Filter-Training.git'

# Auto-detect repo location
if not (REPO_DIR / 'training' / 'train.py').exists():
    matches = list(KAGGLE_WORKING.glob('*/training/train.py'))
    if matches:
        REPO_DIR = matches[0].parents[1]
    else:
        clone_dir = KAGGLE_WORKING / 'Modern-AI-Log-Filter-Training'
        if not clone_dir.exists():
            print('Cloning repo...')
            subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, str(clone_dir)], check=True)
        REPO_DIR = clone_dir

if not (REPO_DIR / 'training' / 'train.py').exists():
    raise FileNotFoundError('Could not find training/train.py. Upload or clone the repo first.')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)

## 2. Install dependencies

In [ ]:
# Kaggle pre-installs PyTorch, but we need the HF/ONNX training stack.
# Install the scientific stack as one compatible set to avoid numpy/pandas ABI mismatch.
%pip install -q --upgrade --force-reinstall --no-deps torch --index-url https://download.pytorch.org/whl/cu126
%pip uninstall -y -q torchvision torchaudio || true
%pip install -q --upgrade --force-reinstall numpy==2.2.6 pandas==2.2.3 scipy==1.15.3 scikit-learn==1.6.1
%pip install -q --upgrade transformers datasets evaluate accelerate 'optimum[onnxruntime]' onnx onnxruntime kagglehub
import numpy as np, pandas as pd
print('numpy', np.__version__, 'pandas', pd.__version__)


## 2b. Kaggle compatibility patch

This notebook patches the cloned training script at runtime so ModernBERT does not invoke the TorchDynamo/Triton compile path on Kaggle GPUs. It also limits training to one visible GPU because the current custom weighted-loss trainer expects an unwrapped model.


In [ ]:
# Kaggle compatibility: single GPU and disable ModernBERT torch.compile/Triton path.
script_path = REPO_DIR / 'training' / 'train_transformer.py'
text = script_path.read_text()
shim = '''
# Kaggle P100/Triton compatibility shim injected by the notebook.
import os as _logfilter_os
_logfilter_os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
_logfilter_os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
try:
    import torch as _logfilter_torch
    def _logfilter_no_compile(fn=None, *args, **kwargs):
        if fn is None:
            return lambda inner: inner
        return fn
    _logfilter_torch.compile = _logfilter_no_compile
except Exception:
    pass
'''.lstrip()
if 'Kaggle P100/Triton compatibility shim injected by the notebook' not in text:
    marker = 'from __future__ import annotations\n'
    if marker in text:
        text = text.replace(marker, marker + shim, 1)
    else:
        text = shim + text
    script_path.write_text(text)
print('Injected single-GPU TorchDynamo/Triton compatibility shim into', script_path)


## 3. Attach HDFS TraceBench dataset

In [ ]:
PREPROCESSED_DIR = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'

def has_trace_files(path: Path) -> bool:
    return (path / 'normal_trace.csv').exists() and (path / 'failure_trace.csv').exists()

if not has_trace_files(PREPROCESSED_DIR):
    input_root = Path('/kaggle/input')
    if input_root.exists():
        # Search Kaggle input for the dataset
        candidates = [p for p in input_root.rglob('preprocessed') if has_trace_files(p)]
        if not candidates:
            candidates = [p for p in input_root.iterdir() if p.is_dir() and has_trace_files(p)]
    else:
        candidates = []
    
    if not candidates:
        # Fallback: download via kagglehub
        import kagglehub
        downloaded = Path(kagglehub.dataset_download('jacobvalor/hdfs-tracebench-preprocessed-logs'))
        print('Downloaded dataset fallback:', downloaded)
        candidates = [downloaded] if has_trace_files(downloaded) else [p for p in downloaded.rglob('preprocessed') if has_trace_files(p)]
    
    if not candidates:
        raise FileNotFoundError('No dataset with normal_trace.csv and failure_trace.csv found.')
    
    source = candidates[0]
    PREPROCESSED_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not PREPROCESSED_DIR.exists():
        os.symlink(source, PREPROCESSED_DIR, target_is_directory=True)
    print('Linked dataset:', source, '->', PREPROCESSED_DIR)
else:
    print('Dataset already available:', PREPROCESSED_DIR)

## 4. Select base model

In [ ]:
# Default: use the published Cisco SecureBERT2.0-base model.
# If a log-adapted MLM artifact exists, prefer the full artifact path first,
# then fall back to the sampled MLM artifact.

MLM_MODEL_CANDIDATES = [
    REPO_DIR / 'models' / 'securebert2-logs-mlm' / 'final',
    REPO_DIR / 'models' / 'securebert2-logs-mlm-sample' / 'final',
]

def has_mlm_artifact(path: Path) -> bool:
    return (path / 'config.json').exists() and (path / 'tokenizer.json').exists() and (
        (path / 'model.safetensors').exists() or (path / 'pytorch_model.bin').exists()
    )

MODEL_ID = next(
    (str(path) for path in MLM_MODEL_CANDIDATES if has_mlm_artifact(path)),
    'cisco-ai/SecureBERT2.0-base',
)
print('Tier-2 base model:', MODEL_ID)


## 5. Sanity preview (text windows)

In [ ]:
sys.path.insert(0, str(REPO_DIR))
from training.text_dataset import build_windows

texts, labels, stats = build_windows(sample_normal=1, sample_failure=1, random_state=42)
print(json.dumps(stats, indent=2))
for text, label in zip(texts, labels):
    kind = 'normal' if label == 0 else 'failure'
    print(f'\n--- {kind} window: label={label}, chars={len(text)} ---')
    print(text[:1200])
    if len(text) > 1200:
        print('... (truncated)')

## 6. Sampled training run (VERIFY FIRST)

**Run this first.** It trains on 50K normal + 10K failure samples for 1 epoch to verify
the environment, GPU, tokenizer, and ONNX export work correctly.

Expected time: ~15-25 minutes on T4 GPU.

In [ ]:
sample_cmd = [
    sys.executable,
    'training/train_transformer.py',
    '--model-id', MODEL_ID,
    '--sample-normal', '50000',
    '--sample-failure', '10000',
    '--epochs', '1',
    '--batch-size', '4',
    '--output-dir', 'models/tier2',
]
print('Command:', ' '.join(sample_cmd))
subprocess.run(sample_cmd, check=True)

## 7. Full training run (AFTER sampled run succeeds)

**Only run this after the sampled run above succeeds.**

Uncomment the cell below by removing the `#` at the start of each line, then run it.
This uses the full dataset (~226K normal + ~30K failure) for 2 epochs.

Expected time: ~2-4 hours on T4 GPU.

In [ ]:
# Uncomment the lines below after the sampled run succeeds:
# full_cmd = [
#     sys.executable,
#     'training/train_transformer.py',
#     '--model-id', MODEL_ID,
#     '--epochs', '2',
#     '--batch-size', '4',
#     '--output-dir', 'models/tier2',
# ]
# print('Command:', ' '.join(full_cmd))
# subprocess.run(full_cmd, check=True)

## 8. Inspect artifacts

In [ ]:
models_dir = REPO_DIR / 'models' / 'tier2'
print('\nArtifacts in', models_dir, ':\n')
for path in sorted(models_dir.glob('*')):
    if path.is_file():
        print(f'  {path.name}: {path.stat().st_size:,} bytes')
    elif path.is_dir():
        size = sum(p.stat().st_size for p in path.rglob('*') if p.is_file())
        print(f'  {path.name}/: {size:,} bytes')

metrics_path = models_dir / 'tier2_metrics.json'
if metrics_path.exists():
    print('\nMetrics:')
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))

## 9. Package artifacts for download

In [ ]:
zip_path = KAGGLE_WORKING / 'logfilter-tier2-artifacts.zip'
include_names = {
    'config.json',
    'model.safetensors',
    'tokenizer.json',
    'tokenizer_config.json',
    'special_tokens_map.json',
    'vocab.json',
    'merges.txt',
    'log_classifier_tier2.onnx',
    'tier2_metrics.json',
    'tier2_label_map.json',
}

required_outputs = {'config.json', 'model.safetensors', 'tokenizer.json', 'tier2_metrics.json', 'tier2_label_map.json'}
existing_outputs = {path.name for path in models_dir.rglob('*') if path.is_file()}
missing_outputs = required_outputs - existing_outputs
if missing_outputs:
    raise FileNotFoundError(f'Missing required files: {sorted(missing_outputs)}')

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(models_dir.rglob('*')):
        if path.is_file() and path.name in include_names:
            zf.write(path, arcname=str(path.relative_to(REPO_DIR)))

print('\nArtifact zip created:', zip_path)
print('Download from the Output tab on the right panel.')

## Next Steps

1. **Download** `logfilter-tier2-artifacts.zip` from the Output tab or with Kaggle CLI.
2. **Extract** it into your local repo root so it restores `models/tier2/`.
3. **Verify Tier-2 ONNX inference** locally with `Tier2Classifier`:
   ```bash
   PYTHONPATH=src python - <<'PY'
   from pathlib import Path
   from logfilter.models.tier2_classifier import Tier2Classifier
   clf = Tier2Classifier(model_dir=Path('models/tier2'))
   print('ready=', clf.is_ready())
   print(clf.predict_proba(['Failed password for root from 10.0.0.5']).tolist())
   PY
   ```
4. **Run the pipeline smoke test**: `PYTHONPATH=src python scripts/smoke_test_pipeline.py`.
